# Customer Retention Analysis

**Problem:** Find the month-over-month customer retention — a customer is considered "retained" in a given month if they made a purchase in the **immediately preceding month** as well. If the gap between purchases is more than 1 month, it does **not** count as retention.

**Approach:**
1. For each customer, find the previous order date using `LAG` partitioned by `cust_id`
2. Compute the month difference between current and previous order dates
3. Flag a row as retention (`1`) if the month difference is exactly `1`, otherwise `0`
4. Group by month and sum the retention flags to get total retained customers per month

**Solutions below:** SQL (Cell 3) and PySpark (Cell 6)

In [0]:
%sql
-- ============================================================
-- Setup: Create & Populate Transactions Table
-- Columns: order_id, cust_id, order_date, amount
-- Sample data: 5 customers with purchases in Jan/Feb 2020
-- ============================================================

create table b_sql.b_practice.transactions(
order_id int,
cust_id int,
order_date date,
amount int
);
insert into b_sql.b_practice.transactions values 
(1,1,'2020-01-15',150)
,(2,1,'2020-02-10',150)
,(3,2,'2020-01-16',150)
,(4,2,'2020-02-25',150)
,(5,3,'2020-01-10',150)
,(6,3,'2020-02-20',150)
,(7,4,'2020-01-20',150)
,(8,5,'2020-02-20',150)
;

In [0]:
%sql
-- Preview: Transactions Data
-- Verify inserted rows and inspect order dates and amounts
select * from b_sql.b_practice.transactions

In [0]:
%sql
-- ============================================================
-- SQL Solution: Month-over-Month Customer Retention
-- 1. retension CTE → LAG to get previous order date per customer,
--    compute month_diff between current & previous order months
-- 2. flag CTE     → set retention = 1 when month_diff = 1, else 0
-- 3. Final SELECT  → group by month, sum retention flags
-- ============================================================
-- retention means: purchased in the immediately preceding month
-- gap of 2+ months does NOT count as retention

with retension as(select *,lag(order_date,1) over(partition by cust_id order by order_date) prev_date,months_between(date_trunc('month',order_date),date_trunc('month',lag(order_date,1) over(partition by cust_id order by order_date))) as month_diff from b_sql.b_practice.transactions
),flag as(
select *,case when month_diff=1 then 1 else 0  end as retention from retension )
select month(order_date) as month,sum(retention) as total_retention from flag  group by month(order_date) order by month(order_date)

In [0]:
# PySpark: Load Transactions into DataFrame
# Read transactions table from Unity Catalog and preview
df_transaction=spark.read.table("b_sql.b_practice.transactions")
df_transaction.display()

In [0]:
# ============================================================
# PySpark Setup: Import Functions, Types & Window
# Imports for the PySpark retention solution below
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
# ============================================================
# PySpark Solution: Month-over-Month Customer Retention
# 1. LAG → get previous_order per customer (partition by cust_id)
# 2. months_between → compute diff in months
# 3. flag = 1 when diff = 1 (consecutive month), else 0
# 4. Group by year/month, sum flags for total retention
# ============================================================
df_transactions=(df_transaction
                .withColumn("previous_order",
                    lag("order_date",1).over(Window.partitionBy("cust_id").orderBy("order_date")
                    ))
                .withColumn("diff",
                            months_between(date_trunc('month',col("order_date")), date_trunc('month',col("previous_order")))
                            )
                .withColumn("flag",when(col("diff")==1,1).otherwise(0))
)
                
df_transactions.display()
df_final=df_transactions.groupBy(year(col("order_date")),month(col("order_date"))).agg(sum("flag").alias("total_retention"))
df_final.display()